# FinDrama LOB Pretraining

This is the only supported Colab notebook. It installs the CUDA/Mamba stack, prepares the Polymarket data bundle, and runs `src/train_lob.py`.

In [ ]:
# Edit these before running.
REPO_URL = "https://github.com/Ruuudy1/FinDrama.git"
BRANCH = "master"
PROJECT_DIR = "/content/Drama"

# Optional. Leave blank to auto-search Drive for a .zip containing the data bundle.
DATA_ZIP = ""

# Use True for a dependency/data smoke test before the full run.
SMOKE_TEST = False
HOURS_TRAIN = 6
HOURS_VAL = 1
MAX_STEPS = 20 if SMOKE_TEST else 20000

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR])
os.chdir(PROJECT_DIR)
print("Repo ready:", os.getcwd())

In [ ]:
import subprocess, sys, os

os.chdir(PROJECT_DIR)

def run(cmd):
    print("\n$", cmd)
    subprocess.check_call(cmd, shell=True)

run("pip install -q --upgrade pip")
run("pip install -q packaging ninja setuptools==69.5.1")
run("pip install -q torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121")
run("pip install -q causal-conv1d==1.2.0.post2 --no-build-isolation")
run("pip install -q mamba-ssm==1.2.0.post1 --no-build-isolation")
# requirements.txt intentionally excludes torch, causal-conv1d, and mamba-ssm;
# those CUDA packages were installed above in the required order.
run("pip install -r requirements.txt")

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os, shutil, tarfile, zipfile

project = Path(PROJECT_DIR)
data_dir = project / "data"
train_dir = data_dir / "train"
val_dir = data_dir / "validation"

def find_data_zip():
    if DATA_ZIP:
        p = Path(DATA_ZIP)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    root = Path('/content/drive/MyDrive')
    zips = sorted(root.rglob('*.zip'), key=lambda p: p.stat().st_size, reverse=True)
    if not zips:
        raise FileNotFoundError('No .zip files found under /content/drive/MyDrive; set DATA_ZIP explicitly.')
    return zips[0]

def extract_bundle():
    if (train_dir / 'polymarket.db').exists() and (val_dir / 'polymarket.db').exists():
        print('Data already prepared at', data_dir)
        return
    zip_path = find_data_zip()
    raw = Path('/content/findrama_data_raw')
    if raw.exists():
        shutil.rmtree(raw)
    raw.mkdir(parents=True)
    print('Extracting', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(raw)
    for archive in list(raw.rglob('*.tar')) + list(raw.rglob('*.tar.gz')) + list(raw.rglob('*.tgz')):
        out = raw / archive.stem.replace('.tar', '')
        out.mkdir(exist_ok=True)
        print('Extracting', archive)
        with tarfile.open(archive) as tf:
            tf.extractall(out)
    split_dirs = [p.parent for p in raw.rglob('polymarket.db')]
    if not split_dirs:
        raise RuntimeError('Could not find polymarket.db after extraction.')
    train_src = next((p for p in split_dirs if 'train' in str(p).lower()), split_dirs[0])
    val_src = next((p for p in split_dirs if 'validation' in str(p).lower() or 'val' in str(p).lower()), None)
    if val_src is None:
        raise RuntimeError('Could not identify validation split; set up data/validation manually.')
    data_dir.mkdir(exist_ok=True)
    shutil.rmtree(train_dir, ignore_errors=True)
    shutil.rmtree(val_dir, ignore_errors=True)
    shutil.copytree(train_src, train_dir)
    shutil.copytree(val_src, val_dir)
    print('Prepared train:', train_dir)
    print('Prepared validation:', val_dir)

extract_bundle()

In [ ]:
import subprocess, os, sys

os.chdir(PROJECT_DIR)
cmd = [
    sys.executable, '-B', 'src/train_lob.py',
    '--hours-train', str(HOURS_TRAIN),
    '--hours-val', str(HOURS_VAL),
    '--JointTrainAgent.SampleMaxSteps', str(MAX_STEPS),
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import shutil

src = Path(PROJECT_DIR) / 'saved_models' / 'lob'
dst = Path('/content/drive/MyDrive/findrama_checkpoints')
dst.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copytree(src, dst / 'lob', dirs_exist_ok=True)
    print('Backed up to', dst / 'lob')
else:
    print('No checkpoint directory found yet:', src)